# Model Performance and Degradation Trend Comparison

This notebook visualizes the degradation trends for different models. It compares the Signal RMS with the Model's Reconstruction Error.

In [16]:
import os
import sys
import yaml
import torch
import pandas as pd
import numpy as np

# 1. Xác định thư mục gốc của dự án (Project Root)
notebook_dir = os.getcwd()
# Dùng thư mục cha của cha (../../) từ src/notebooks để về tmp_project
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Thiết lập các đường dẫn tuyệt đối để tránh lỗi "No such file"
config_path = os.path.join(project_root, 'configs', 'default.yaml')
models_dir = os.path.join(project_root, 'results', 'models')

# 3. Nạp cấu hình
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# 4. Xác định đường dẫn dữ liệu đã xử lý
processed_dir = os.path.join(project_root, 'data', 'processed')

print(f"Project Root: {project_root}")
print(f"Config loaded from: {config_path}")
print(f"Data directory: {processed_dir}")

# Khởi tạo Dataset
test_dataset = B02Dataset(
    processed_dir, 
    lookback=config['data'].get('lookback', 512), 
    horizon=config['data'].get('forecast_len', 128), 
    stride=128, 
    split='test', 
    train_ratio=config['data'].get('train_ratio', 0.5), 
    skip_ratio=config['data'].get('skip_ratio', 0.1)
)


Project Root: /mnt/f/APPS_PJ/mamba-forecast-ad/tmp_project
Config loaded from: /mnt/f/APPS_PJ/mamba-forecast-ad/tmp_project/configs/default.yaml
Data directory: /mnt/f/APPS_PJ/mamba-forecast-ad/tmp_project/data/processed


In [17]:
def get_model(model_name, config, device):
    lookback = config['data'].get('lookback', 512)
    horizon = config['data'].get('forecast_len', 128)
    
    if "hybrid" in model_name.lower():
        model = HybridMambaCNN({
            'model': {
                'mamba_version': 1,
                'mamba_d_model': 64, 'mamba_n_layer': 4,
                'mamba_d_state': 16, 'mamba_d_conv': 4, 'mamba_expand': 2,
                'forecast_len': horizon, 'patch_size': 64, 'stride': 32,
                'in_channels': 2, 'lookback': lookback,
                'decomp_kernel': 25, 'use_multiscale': True,
            },
            'data': {'patch_size': 64, 'stride': 32, 'lookback': lookback}
        })
    elif "mambats" in model_name.lower():
        # Note: Set use_casual_conv=True if weights were trained with it ON
        mamba_config = MambaTSConfig(
            in_channels=2, lookback=lookback, forecast_len=horizon,
            patch_size=64, stride=32, d_model=64, n_layers=4,
            dropout=0.2, VPT_mode=1, ATSP_solver='SA', use_casual_conv=True
        )
        model = MambaTSOfficial(mamba_config)
    else:
        raise ValueError(f"Unknown model type for {model_name}")
        
    return model

In [18]:
def plot_degradation_trend(model, model_name, dataset, config, device):
    lookback = config['data'].get('lookback', 512)
    horizon = config['data'].get('forecast_len', 128)
    
    # Dùng đường dẫn tuyệt đối dựa trên project_root
    processed_dir = os.path.join(project_root, "data/processed")
    oc_raw_path = os.path.join(project_root, "data/raw/B02/B02_operatingConditions.csv")
    
    if not os.path.exists(oc_raw_path):
        print(f"Error: OC file not found at {oc_raw_path}")
        return
    oc_df_raw = pd.read_csv(oc_raw_path)
    
    # Chuẩn hóa OC để khớp với model (Dùng stats từ dataset đã nạp)
    oc_cols = oc_df_raw.columns[1:]
    oc_values = oc_df_raw[oc_cols].values
    if hasattr(dataset, 'oc_stats') and dataset.oc_stats is not None:
        oc_values = (oc_values - dataset.oc_stats['mean']) / dataset.oc_stats['std']
    
    # Danh sách file thực tế trong thư mục processed để khớp index
    all_files_ordered = sorted([f for f in os.listdir(processed_dir) if f.endswith('.pt')])
    test_files = [dataset.files[idx] for idx in sorted(list(set([s[0] for s in dataset.samples])))]
    all_files_ordered = sorted([f for f in os.listdir(processed_dir) if f.endswith('.pt')])
    file_indices, avg_errors, file_rms = [], [], []
    
    model.eval()
    with torch.no_grad():
        for i, f_name in enumerate(tqdm(test_files, desc=f"Scanning {model_name}")):
            f_path = os.path.join(processed_dir, f_name)
            signal = torch.load(f_path, map_location='cpu', weights_only=True)
            
            # Khớp index từ file name sang hàng trong CSV
            global_idx = all_files_ordered.index(f_name)
            oc = torch.from_numpy(oc_values[global_idx].astype('float32')).unsqueeze(0).to(device)
            
            n_samples = signal.shape[1]
            win_starts = np.linspace(0, n_samples - lookback - horizon, 5, dtype=int)
            errs = []
            for start in win_starts:
                x = signal[:, start:start+lookback].unsqueeze(0).to(device)
                y = signal[:, start+lookback:start+lookback+horizon].unsqueeze(0).to(device)
                
                if "mambats" in model_name.lower():
                    y_pred = model(x, oc)
                else:
                    # Tính stats cho HybridMamba
                    eps = 1e-8
                    mean = x.mean(dim=-1, keepdim=True)
                    std  = x.std(dim=-1, keepdim=True) + eps
                    rms_val  = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True))
                    p2p  = x.max(dim=-1, keepdim=True)[0] - x.min(dim=-1, keepdim=True)[0]
                    skew = torch.mean(((x - mean) / std) ** 3, dim=-1, keepdim=True)
                    kurt = torch.mean(((x - mean) / std) ** 4, dim=-1, keepdim=True)
                    crest = torch.max(torch.abs(x), dim=-1, keepdim=True)[0] / (rms_val + eps)
                    shape = rms_val / (torch.mean(torch.abs(x), dim=-1, keepdim=True) + eps)
                    stats = torch.cat([mean, std, rms_val, p2p, skew, kurt, crest, shape], dim=-1)
                    y_pred = model(x, stats)
                    
                errs.append(calculate_anomaly_score(y, y_pred, metric='mse', normalized=True).item())
            
            avg_errors.append(np.mean(errs))
            file_rms.append(dataset.file_rms[f_name])
            file_indices.append(i)

    # Plotting (Giữ nguyên phần vẽ đồ thị...)
    fig, ax1 = plt.subplots(figsize=(15, 5))
    ax1.plot(file_indices, file_rms, color='tab:blue', label='Signal RMS')
    ax2 = ax1.twinx()
    ax2.plot(file_indices, avg_errors, color='tab:red', alpha=0.6)
    ax2.set_yscale('log')
    plt.title(f"Degradation Trend: {model_name}")
    plt.show()

In [19]:
# 1. Setup paths and config
config_path = "../../configs/default.yaml"
models_dir = "../../results/models"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# 2. Initialize Dataset (Test Split)
lookback = config['data'].get('lookback', 512)
horizon = config['data'].get('forecast_len', 128)
processed_dir = "../../" + config['data']['processed_dir']

test_dataset = B02Dataset(
    processed_dir, lookback, horizon, stride=128, 
    split='test', 
    train_ratio=config['data'].get('train_ratio', 0.5), 
    skip_ratio=config['data'].get('skip_ratio', 0.1)
)

# 3. Run evaluation
for model_file in sorted(os.listdir(models_dir)):
    if model_file.endswith(".pth"):
        model_name = model_file.replace("_best.pth", "").upper()
        model_path = os.path.join(models_dir, model_file)
        
        print(f"\n>>> Processing {model_name}...")
        try:
            model = get_model(model_name, config, device)
            model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
            model.to(device)
            
            plot_degradation_trend(model, model_name, test_dataset, config, device)
        except Exception as e:
            print(f"Error plotting {model_name}: {e}")


>>> Processing MAMBA1_HYBRID...
Error plotting MAMBA1_HYBRID: Error(s) in loading state_dict for HybridMambaCNN:
	size mismatch for trend_head.weight: copying a param with shape torch.Size([128, 512]) from checkpoint, the shape in current model is torch.Size([128, 4096]).

>>> Processing MAMBATS_OFFICIAL...
Error plotting MAMBATS_OFFICIAL: Error(s) in loading state_dict for MambaTSOfficial:
	size mismatch for head.linear.weight: copying a param with shape torch.Size([128, 960]) from checkpoint, the shape in current model is torch.Size([128, 8128]).
